# F1Compiler 🏎️

**Lenguaje de programación basado en telemetría y estrategias de Fórmula 1.**

Este notebook ejecuta todas las fases del compilador interactuando con los archivos locales del proyecto, preservando su arquitectura modular.

### Preparación del Entorno
Antes de ejecutar las celdas de este Notebook, asegúrate de haber seguido las instrucciones de instalación detalladas en el **`README.md`**.

Esto incluye:
1. Instalar **Graphviz** en tu sistema.
2. Crear y activar tu **entorno virtual** (`.venv`).
3. Instalar las dependencias de Python con `pip install -r requirements.txt`.

### Instrucciones de Ejecución
1. Asegúrate de estar corriendo el kernel de Jupyter utilizando tu entorno `.venv`.
2. Ejecuta las celdas secuencialmente de arriba a abajo.
3. Los archivos fuente evaluados están en `examples/` y los módulos en `src/tools/`.


## Fase 1: Análisis Léxico
Ejecuta el lexer sobre el programa de ejemplo, mostrando la tabla detallada de tokens reconocidos.

In [30]:
!python ejecutar_lexer.py examples/estrategia_carrera.txt


Programa de ejemplo definido (1452 caracteres).
ANÁLISIS LÉXICO — Lista completa de tokens
Num   Tipo                      Texto                          Línea    Columna
--------------------------------------------------------------------------------
1     GREEN_LIGHT               'green_light'                  6        0
4     TYPE_TELEMETRY            'telemetry'                    8        4
39    ID                        'base_pace'                    8        14
29    ASSIGN                    '='                            8        24
36    NUM_FLOAT                 '85.5'                         8        26
30    SEMI                      ';'                            8        30
6     TYPE_COMPOUND             'compound'                     9        4
39    ID                        'current_tyre'                 9        13
29    ASSIGN                    '='                            9        26
38    STRING                    '"Soft"'                       9        28
3

## Fase 2: Análisis Sintáctico (AST)
Genera el Árbol de Sintaxis Abstracta (AST) usando Graphviz para visualizar las relaciones del código fuente.

In [39]:
!python src/tools/herramientas_ast.py examples/estrategia_carrera.txt ast_carrera


Generando telemetría visual del AST...
Árbol en formato texto:
(program green_light (statement (declaracion (tipo telemetry) base_pace = (expresion 85.5) ;)) (statement (declaracion (tipo compound) current_tyre = (expresion "Soft") ;)) (statement (impresion engineer_says ( (expresion "--- Starting Race Simulation ---") ) ;)) (statement (ciclo start_new_stint ( (asignacion current_lap = (expresion 1) ;) up_to_lap (expresion 3) ) { (statement (impresion engineer_says ( (expresion "Lap number:") ) ;)) (statement (impresion engineer_says ( (expresion current_lap) ) ;)) (statement (condicional if_pitwall_says ( (expresion (expresion (expresion current_lap) matched_delta (expresion 2)) and_drs_open (expresion track_clear)) ) { (statement (impresion engineer_says ( (expresion "Pushing! DRS Enabled") ) ;)) (statement (asignacion base_pace = (expresion (expresion base_pace) catch_slipstream (expresion 1.05)) ;)) } otherwise_box { (statement (asignacion base_pace = (expresion (expresion base_pac

In [40]:
!python src/tools/herramientas_ast.py examples/estrategia_lluvia.txt ast_lluvia

Generando telemetría visual del AST...
Árbol en formato texto:
(program green_light (statement (declaracion (tipo telemetry) delta_time = (expresion 0.0) ;)) (statement (declaracion (tipo compound) start_tyre = (expresion "Medium") ;)) (statement (declaracion (tipo steward_decision) rain_expected = (expresion track_clear) ;)) (statement (impresion engineer_says ( (expresion "--- Starting Wet Race Simulation ---") ) ;)) (statement (impresion engineer_says ( (expresion "Starting on:") ) ;)) (statement (impresion engineer_says ( (expresion start_tyre) ) ;)) (statement (ciclo start_new_stint ( (asignacion current_lap = (expresion 1) ;) up_to_lap (expresion 4) ) { (statement (condicional if_pitwall_says ( (expresion (expresion current_lap) matched_delta (expresion 3)) ) { (statement (impresion engineer_says ( (expresion "Radar shows rain approaching!") ) ;)) (statement (asignacion rain_expected = (expresion red_flag) ;)) } otherwise_box { (statement (impresion engineer_says ( (expresion "Tr

## Fase 3: Análisis Semántico
Evalúa las reglas semánticas (Dirección de Carrera) detectando variables duplicadas, no declaradas y sin uso.

In [32]:
# Análisis semántico
!python src/tools/herramientas_semanticas.py examples/estrategia_carrera.txt


REPORTE DE DIRECCIÓN DE CARRERA (Análisis Semántico)
REPORTE DE DIRECCIÓN DE CARRERA (Análisis Semántico)
Black Flag (Línea 14): Intento de reasignar 'current_lap', pero no existe en telemetría.
Black Flag (Línea 16): Se intentó leer 'current_lap', pero no está registrado.
Black Flag (Línea 19): Se intentó leer 'current_lap', pero no está registrado.
Warning (Línea 9): El parámetro 'current_tyre' fue declarado pero nunca se utilizó en la estrategia.

Tabla de Símbolos Final:
  - base_pace: {'tipo': 'telemetry', 'linea': 8, 'usado': True}
  - current_tyre: {'tipo': 'compound', 'linea': 9, 'usado': False}
  - final_wear: {'tipo': 'telemetry', 'linea': 28, 'usado': True}
  - is_raining: {'tipo': 'steward_decision', 'linea': 33, 'usado': True}


In [33]:
# Ejemplo de análisis semántico con un código de prueba diferente
!python src/tools/herramientas_semanticas.py examples/estrategia_lluvia.txt

REPORTE DE DIRECCIÓN DE CARRERA (Análisis Semántico)
REPORTE DE DIRECCIÓN DE CARRERA (Análisis Semántico)
Black Flag (Línea 18): Intento de reasignar 'current_lap', pero no existe en telemetría.
Black Flag (Línea 19): Se intentó leer 'current_lap', pero no está registrado.
Warning (Línea 31): El parámetro 'new_tyre' fue declarado pero nunca se utilizó en la estrategia.

Tabla de Símbolos Final:
  - delta_time: {'tipo': 'telemetry', 'linea': 9, 'usado': True}
  - start_tyre: {'tipo': 'compound', 'linea': 10, 'usado': True}
  - rain_expected: {'tipo': 'steward_decision', 'linea': 11, 'usado': True}
  - new_tyre: {'tipo': 'compound', 'linea': 31, 'usado': False}
  - undercut_loss: {'tipo': 'telemetry', 'linea': 35, 'usado': True}
  - remaining_laps: {'tipo': 'laps', 'linea': 43, 'usado': True}


## Fase 4: Generación de Código (Transpilación) y Ejecución
Traduce el código fuente F1Compiler a Python puro y lo ejecuta dinámicamente con exec().

In [34]:
# Ejemplo de generación de código principal
!python generador_codigo.py examples/estrategia_carrera.txt

Leyendo programa desde: examples/estrategia_carrera.txt
CÓDIGO TRADUCIDO (PYTHON):
# ==========================================
# CÓDIGO GENERADO AUTOMÁTICAMENTE (F1Compiler)
# ==========================================
import math

# --- Funciones Nativas de Telemetría ---
def tyre_wear_calc(x): return math.sqrt(x)
def calc_steering(x): return math.cos(math.radians(x))
def deploy_ers(x, y): return math.pow(x, y)
def calculate_undercut_delta(x): return math.log(x)

# --- Estrategia Principal ---
base_pace = 85.5
current_tyre = "Soft"
print("--- Starting Race Simulation ---")
for current_lap in range(int(1), int(3) + 1):
    print("Lap number:")
    print(current_lap)
    if current_lap == 2 and True:
        print("Pushing! DRS Enabled")
        base_pace = base_pace * 1.05
    else:
        base_pace = base_pace + 0.1
final_wear = tyre_wear_calc(base_pace)
print("Final Telemetry:")
print(final_wear)
is_raining = True
while not is_raining:
    print("Track is dry. Keeping the pace.")
 

In [35]:
# Ejemplo de generación de código con un código de prueba diferente
!python generador_codigo.py examples/estrategia_lluvia.txt

Leyendo programa desde: examples/estrategia_lluvia.txt
CÓDIGO TRADUCIDO (PYTHON):
# ==========================================
# CÓDIGO GENERADO AUTOMÁTICAMENTE (F1Compiler)
# ==========================================
import math

# --- Funciones Nativas de Telemetría ---
def tyre_wear_calc(x): return math.sqrt(x)
def calc_steering(x): return math.cos(math.radians(x))
def deploy_ers(x, y): return math.pow(x, y)
def calculate_undercut_delta(x): return math.log(x)

# --- Estrategia Principal ---
delta_time = 0.0
start_tyre = "Medium"
rain_expected = True
print("--- Starting Wet Race Simulation ---")
print("Starting on:")
print(start_tyre)
for current_lap in range(int(1), int(4) + 1):
    if current_lap == 3:
        print("Radar shows rain approaching!")
        rain_expected = False
    else:
        print("Track is still dry, keep pushing.")
        delta_time = delta_time + 0.2
if rain_expected == False:
    print("Box, box! Changing to Wet tyres.")
    new_tyre = "Wet"
    delta_tim

## Casos de Prueba Adicionales
Ejecuta diferentes escenarios de código de ejemplo para validar los distintos componentes lógicos y cíclicos de la gramática.

In [36]:
!python casos_prueba.py


  Prueba 1 — Tipos de Datos y Setup
Num   Tipo                      Texto                          Línea    Columna
--------------------------------------------------------------------------------
1     GREEN_LIGHT               'green_light'                  2        0
6     TYPE_COMPOUND             'compound'                     4        4
39    ID                        'current_tyre'                 4        13
29    ASSIGN                    '='                            4        26
38    STRING                    '"Hard"'                       4        28
30    SEMI                      ';'                            4        34
4     TYPE_TELEMETRY            'telemetry'                    5        4
39    ID                        'fuel_load'                    5        14
29    ASSIGN                    '='                            5        24
36    NUM_FLOAT                 '105.5'                        5        26
30    SEMI                      ';'                    